# P2-G6_1: Strict P3→P4 Run-up Summary

이 노트북은 `strict-clean` P2-G4 연구연쇄에서 이미 실행된 P3/P4 산출물을 읽어
P6 진입 전 상태를 고정하는 read-only 요약 노트북이다.

핵심 질문은 세 가지다.

1. P3-S OOF Grade Residual은 전체 strict A비율 표본을 덮고 있는가?
2. P4에서 RAW_A와 OOF residual은 건강보험 취업률·대학원 진학률에 추가정보를 제공하는가?
3. P5 residual heterogeneity와 P6 residual topology로 넘어갈 때 어떤 경고를 유지해야 하는가?

주의: 이 노트북은 P3/P4 모형을 재적합하지 않는다. 기존 strict 산출물의 lineage, 수치, 해석 안전장치를 한곳에 묶는다.


In [1]:
from pathlib import Path
import json
import hashlib
import platform
import subprocess
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)


def find_repo_root() -> Path:
    """현재 실행 위치가 달라도 P3/P4 strict 상태 파일을 기준으로 repo root를 찾는다."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "workbook/p2/p2_4/P3_P4_CONFIRMATORY_STATUS.json").exists():
            return candidate.resolve()
    fallback = Path("/home/sieg/projects-wsl/SBS_dataScience")
    if (fallback / "workbook/p2/p2_4/P3_P4_CONFIRMATORY_STATUS.json").exists():
        return fallback.resolve()
    raise FileNotFoundError("P3/P4 confirmatory status file was not found from current working directory.")


ROOT = find_repo_root()
OUT_ROOT = ROOT / "workbook/p2/P2_6"
NOTEBOOK_PATH = OUT_ROOT / "P2_G6_1.ipynb"
for subdir in ["artifacts", "figures", "qa", "reports", "logs"]:
    (OUT_ROOT / subdir).mkdir(parents=True, exist_ok=True)


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def git_commit(root: Path) -> str:
    try:
        out = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            cwd=root,
            check=False,
            capture_output=True,
            text=True,
        )
        return out.stdout.strip() or "UNKNOWN"
    except Exception:
        return "UNKNOWN"


def file_record(path: Path, label: str | None = None) -> dict:
    """manifest용 파일 기록. 표 형태 파일은 shape도 함께 남긴다."""
    record = {
        "label": label or path.name,
        "path": str(path.relative_to(ROOT)),
        "exists": path.exists(),
    }
    if not path.exists():
        return record
    record.update(
        {
            "size_bytes": path.stat().st_size,
            "sha256": sha256_file(path),
            "mtime": datetime.fromtimestamp(path.stat().st_mtime).isoformat(),
        }
    )
    try:
        if path.suffix == ".csv":
            record["shape"] = list(pd.read_csv(path).shape)
        elif path.suffix == ".parquet":
            record["shape"] = list(pd.read_parquet(path).shape)
        elif path.suffix == ".json":
            obj = read_json(path)
            record["shape"] = [len(obj)] if hasattr(obj, "__len__") else []
    except Exception as exc:
        record["shape_error"] = repr(exc)
    return record


ENV = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "working_directory": str(Path.cwd()),
    "repo_root": str(ROOT),
    "git_commit": git_commit(ROOT),
    "execution_timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
display(pd.DataFrame([ENV]).T.rename(columns={0: "value"}))

,value
python,3.12.3
platform,Linux-6.18.33.2-microsoft-standard-WSL2-x86_64...
pandas,3.0.3
numpy,2.5.0
working_directory,/home/sieg/projects-wsl/SBS_dataScience/workbo...
repo_root,/home/sieg/projects-wsl/SBS_dataScience
git_commit,5b1a3d54266d881a839ad9a3cec750da66e94bc7
execution_timestamp_utc,2026-07-13T08:31:57.808502+00:00


## V00. 시각화 실행 헬퍼

- 시각화 목적: P3/P4 strict 결과를 다시 학습하지 않고, 모델 구조와 결과 해석을 읽는 그림만 저장한다.
- 사용할 데이터: 이후 셀에서 로드되는 P3/P4 CSV·parquet 산출물.
- 필요한 전처리: 공통 figure 저장 함수와 관찰/원인/제한/결론 해석 템플릿.
- 코드 셀 설계: 모든 추가 그림은 `workbook/p2/P2_6/figures` 아래 저장하고 manifest에 누적한다.
- 그래프 해석 포인트: figure 자체보다 figure가 답하는 모델 질문을 우선한다.
- 학생이 자주 하는 오해: 새 그림이 새 모델 적합을 의미한다고 착각하는 것.
- 체크포인트 질문: 이 노트북은 모델을 다시 학습하는가, 아니면 strict 산출물을 읽는가?

In [2]:
# P2-G6 visual development helper.
# 기존 P3/P4 strict 산출물을 다시 적합하지 않고, 결과 구조를 읽는 그림만 만든다.
VISUAL_FIGURE_RECORDS = []


def save_visual_figure(fig, filename: str, block_id: str, question: str, data_used: str):
    path = OUT_ROOT / "figures" / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=180, bbox_inches="tight")
    VISUAL_FIGURE_RECORDS.append(
        {
            "block_id": block_id,
            "figure_path": str(path.relative_to(ROOT)),
            "question": question,
            "data_used": data_used,
        }
    )
    return path


def status_to_score(value) -> float:
    text = str(value).upper()
    if "BLOCKED" in text or "FAIL" in text:
        return 0.0
    if "WARNING" in text or "WARN" in text:
        return 0.5
    if "READY" in text or "PASS" in text or "OK" in text:
        return 1.0
    return 0.25


def display_reading_note(title: str, observation: str, cause: str, limitation: str, conclusion: str):
    display(
        Markdown(
            f"""**{title}**

- 관찰: {observation}
- 원인: {cause}
- 제한: {limitation}
- 결론: {conclusion}
"""
        )
    )

In [3]:
# 이 셀은 P3/P4 strict 실행 산출물의 위치와 존재 여부를 먼저 고정한다.
P3_ROOT = ROOT / "workbook/p2/p2_4/p3_grade_residual_v1_strict"
P4_ROOT = ROOT / "workbook/p2/p2_4/p4_grade_signal_outcomes_v1_strict"

PATHS = {
    "integrated_status": ROOT / "workbook/p2/p2_4/P3_P4_CONFIRMATORY_STATUS.json",
    "integrated_report": ROOT / "workbook/p2/p2_4/P3_P4_CONFIRMATORY_STATUS.md",
    "p3_status": P3_ROOT / "reports/P3_GRADE_RESIDUAL_STATUS.json",
    "p3_manifest": P3_ROOT / "logs/P3_OUTPUT_MANIFEST.json",
    "p3_oof": P3_ROOT / "artifacts/P3_OOF_PERFORMANCE.csv",
    "p3_test": P3_ROOT / "artifacts/P3_LOCKED_TEST_PERFORMANCE.csv",
    "p3_diag": P3_ROOT / "artifacts/P3_RESIDUAL_DIAGNOSTICS.csv",
    "p3_sample": P3_ROOT / "qa/P3_SAMPLE_AUDIT.csv",
    "p3_q_gate": P3_ROOT / "qa/P3_Q_APPROVAL_GATE.csv",
    "p3_full_residual": P3_ROOT / "data/P3_STRUCTURE_GRADE_RESIDUAL_FULL.parquet",
    "p3_nopolicy_residual": P3_ROOT / "data/P3_STRUCTURE_GRADE_RESIDUAL_NOPOLICY.parquet",
    "p4_status": P4_ROOT / "reports/P4_GRADE_SIGNAL_OUTCOME_STATUS.json",
    "p4_manifest": P4_ROOT / "logs/P4_OUTPUT_MANIFEST.json",
    "p4_sample": P4_ROOT / "qa/P4_SAMPLE_AUDIT.csv",
    "p4_coef": P4_ROOT / "artifacts/P4_COEFFICIENT_RESULTS.csv",
    "p4_iv": P4_ROOT / "artifacts/P4_INCREMENTAL_VALUE.csv",
    "p4_d": P4_ROOT / "artifacts/P4_EMPLOYMENT_PROGRESSION_DIFFERENCE.csv",
    "p4_test": P4_ROOT / "artifacts/P4_LOCKED_TEST_METRICS.csv",
    "p4_boot_ci": P4_ROOT / "artifacts/P4_BOOTSTRAP_CI.csv",
    "p4_diag": P4_ROOT / "qa/P4_MODEL_DIAGNOSTICS.csv",
    "p4_holm": P4_ROOT / "qa/P4_MULTIPLE_TEST_CORRECTION.csv",
    "p4_equiv": P4_ROOT / "qa/P4_WITHIN_RAW_EQUIVALENCE_AUDIT.csv",
    "p4_to_p5": P4_ROOT / "artifacts/P4_TO_P5_RESIDUAL_HANDOFF.json",
}

required = [
    "integrated_status",
    "p3_status",
    "p3_manifest",
    "p3_oof",
    "p3_diag",
    "p3_sample",
    "p3_full_residual",
    "p4_status",
    "p4_manifest",
    "p4_sample",
    "p4_coef",
    "p4_iv",
    "p4_d",
    "p4_test",
    "p4_boot_ci",
    "p4_equiv",
    "p4_to_p5",
]
existence = pd.DataFrame(
    [{"key": k, "path": str(PATHS[k].relative_to(ROOT)), "exists": PATHS[k].exists()} for k in PATHS]
)
display(existence)
missing = [k for k in required if not PATHS[k].exists()]
if missing:
    raise FileNotFoundError(f"Required P3/P4 strict artifacts are missing: {missing}")


,key,path,exists
0,integrated_status,workbook/p2/p2_4/P3_P4_CONFIRMATORY_STATUS.json,True
1,integrated_report,workbook/p2/p2_4/P3_P4_CONFIRMATORY_STATUS.md,True
2,p3_status,workbook/p2/p2_4/p3_grade_residual_v1_strict/r...,True
3,p3_manifest,workbook/p2/p2_4/p3_grade_residual_v1_strict/l...,True
4,p3_oof,workbook/p2/p2_4/p3_grade_residual_v1_strict/a...,True
5,p3_test,workbook/p2/p2_4/p3_grade_residual_v1_strict/a...,True
6,p3_diag,workbook/p2/p2_4/p3_grade_residual_v1_strict/a...,True
7,p3_sample,workbook/p2/p2_4/p3_grade_residual_v1_strict/q...,True
8,p3_q_gate,workbook/p2/p2_4/p3_grade_residual_v1_strict/q...,True
9,p3_full_residual,workbook/p2/p2_4/p3_grade_residual_v1_strict/d...,True


In [4]:
# 이 셀은 상태 JSON을 읽어 strict 입력 hash와 P3/P4 준비 상태를 한 표로 만든다.
integrated_status = read_json(PATHS["integrated_status"])
p3_status = read_json(PATHS["p3_status"])
p4_status = read_json(PATHS["p4_status"])
p4_to_p5 = read_json(PATHS["p4_to_p5"])

status_rows = []
for namespace, obj in [("P3", integrated_status["P3"]), ("P4", integrated_status["P4"])]:
    for key, value in obj.items():
        status_rows.append({"namespace": namespace, "status_key": key, "status": value})
df_status = pd.DataFrame(status_rows)

hash_rows = pd.DataFrame(
    [
        {"item": "strict_d08", "sha256": integrated_status.get("strict_d08_sha256")},
        {"item": "p2_to_p3_handoff", "sha256": integrated_status.get("p2_handoff_sha256")},
        {"item": "p3_full_residual", "sha256": p3_status.get("p3_full_residual_sha256")},
        {"item": "p3_nopolicy_residual", "sha256": p3_status.get("p3_nopolicy_residual_sha256")},
        {"item": "p3_notebook_after_run_all", "sha256": p3_status.get("notebook_sha256_after_run_all")},
        {"item": "p4_notebook_reference", "sha256": file_record(P4_ROOT / "p4_grade_signal_outcomes_strict.ipynb")["sha256"]},
        {"item": "p4_to_p5_handoff", "sha256": file_record(PATHS["p4_to_p5"])["sha256"]},
    ]
)

display(df_status)
display(hash_rows)
display(Markdown(
    f"""**Lineage 판정:** strict D08 hash `{integrated_status.get('strict_d08_sha256')}`와
P2→P3 handoff hash `{integrated_status.get('p2_handoff_sha256')}`를 기준으로 P3/P4 산출물을 읽었다.
P3 overall은 `{integrated_status['P3']['P3_OVERALL_STATUS']}`, P4 overall은 `{integrated_status['P4']['P4_OVERALL_STATUS']}`다."""
))


,namespace,status_key,status
0,P3,P3_INPUT_STATUS,READY
1,P3,P3_S_FULL_STATUS,READY
2,P3,P3_S_NOPOLICY_STATUS,READY
3,P3,P3_Q_STATUS,BLOCKED_FEATURE_CONTRACT
4,P3,P3_OOF_COVERAGE_STATUS,READY
5,P3,P3_TEST_STATUS,READY
6,P3,P3_OVERALL_STATUS,READY_WITH_WARNINGS
7,P4,P4_INPUT_STATUS,READY
8,P4,P4_RAW_A_STATUS,READY
9,P4,P4_RESIDUAL_STATUS,READY


,item,sha256
0,strict_d08,5f56e375fd1c0474a5e55652859ae007e2f45becd6d335...
1,p2_to_p3_handoff,2bbc7d64784b7b530d57bb6a2096d14cb11c5879f41a52...
2,p3_full_residual,d8decd39dca42ccd0dc194fa3813ba0036541098eea08d...
3,p3_nopolicy_residual,dc5675d941b17108997e63adb550f6f223911892085cf4...
4,p3_notebook_after_run_all,0bf83493654a3d235321b2c31d1a109473f06d095f7625...
5,p4_notebook_reference,2788ef81cc54686e76adddbb09abd6f023340816c898a3...
6,p4_to_p5_handoff,601c352366a4cd45ac1575daf2836a99dff4711a3f76c8...


**Lineage 판정:** strict D08 hash `5f56e375fd1c0474a5e55652859ae007e2f45becd6d3350ee4c82e21fab8df9b`와
P2→P3 handoff hash `2bbc7d64784b7b530d57bb6a2096d14cb11c5879f41a5208f9ff3b02f4bdddcb`를 기준으로 P3/P4 산출물을 읽었다.
P3 overall은 `READY_WITH_WARNINGS`, P4 overall은 `READY_WITH_WARNINGS`다.

## V01. Status / Lineage Map

- 시각화 목적: P3/P4 strict chain이 같은 입력 hash와 READY/WARNING/BLOCKED 상태를 공유하는지 확인한다.
- 사용할 데이터: `P3_P4_CONFIRMATORY_STATUS.json`, P3/P4 status JSON.
- 필요한 전처리: status 문자열을 ready score로 변환하고 hash를 짧은 anchor로 요약한다.
- 코드 셀 설계: 왼쪽은 readiness matrix, 오른쪽은 lineage hash chain.
- 그래프 해석 포인트: P6는 READY지만 Q branch와 warning은 독립적으로 유지된다.
- 학생이 자주 하는 오해: hash가 같으면 통계 해석까지 자동으로 안전하다고 생각하는 것.
- 체크포인트 질문: `READY_WITH_WARNINGS`는 `READY`와 어떻게 다른가?

In [5]:
# V01. P3/P4 상태와 lineage hash를 한눈에 확인한다.
status_plot = df_status.copy()
status_plot["score"] = status_plot["status"].map(status_to_score)
status_wide = status_plot.pivot(index="namespace", columns="status_key", values="score")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), gridspec_kw={"width_ratios": [1.25, 1]})
im = axes[0].imshow(status_wide.fillna(0.25), aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
axes[0].set_title("P3/P4 readiness status matrix")
axes[0].set_yticks(range(len(status_wide.index)))
axes[0].set_yticklabels(status_wide.index)
axes[0].set_xticks(range(len(status_wide.columns)))
axes[0].set_xticklabels(status_wide.columns, rotation=45, ha="right")
for y, namespace in enumerate(status_wide.index):
    for x, key in enumerate(status_wide.columns):
        raw_status = df_status.loc[
            (df_status["namespace"].eq(namespace)) & (df_status["status_key"].eq(key)), "status"
        ]
        label = raw_status.iloc[0] if len(raw_status) else ""
        axes[0].text(x, y, str(label).replace("_", "\n"), ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=axes[0], fraction=0.035, pad=0.02, label="ready score")

axes[1].axis("off")
hash_text = ["Lineage hash chain"]
for row in hash_rows.itertuples(index=False):
    sha = str(row.sha256)
    hash_text.append(f"{row.item}: {sha[:10]}...{sha[-8:] if len(sha) > 18 else sha}")
axes[1].text(0, 1, "\n".join(hash_text), va="top", ha="left", family="monospace", fontsize=9)
axes[1].set_title("Read-only lineage anchors")
fig.tight_layout()
path = save_visual_figure(
    fig,
    "P2_G6_1_V01_STATUS_LINEAGE.png",
    "V01_STATUS_LINEAGE",
    "P3/P4 strict 상태와 hash가 같은 실행 사슬을 가리키는가?",
    "P3_P4_CONFIRMATORY_STATUS.json, P3/P4 status json",
)
plt.show()
print(f"saved visual: {path.relative_to(ROOT)}")

display_reading_note(
    "V01 해석",
    "P3/P4는 READY 계열 상태지만 P6로 넘길 때 WARNING과 BLOCKED branch를 분리해서 읽어야 한다.",
    "이 노트북은 새 적합이 아니라 이미 잠긴 strict 산출물의 lineage와 상태를 재확인한다.",
    "hash는 재현성 anchor이지 통계적 타당성 자체를 증명하지 않는다.",
    "P6 입력은 준비됐지만 P2-Q/P3-Q 차단과 residual/raw 동등성 경고를 유지한다.",
)

saved visual: workbook/p2/P2_6/figures/P2_G6_1_V01_STATUS_LINEAGE.png


/tmp/ipykernel_183879/3867995522.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**V01 해석**

- 관찰: P3/P4는 READY 계열 상태지만 P6로 넘길 때 WARNING과 BLOCKED branch를 분리해서 읽어야 한다.
- 원인: 이 노트북은 새 적합이 아니라 이미 잠긴 strict 산출물의 lineage와 상태를 재확인한다.
- 제한: hash는 재현성 anchor이지 통계적 타당성 자체를 증명하지 않는다.
- 결론: P6 입력은 준비됐지만 P2-Q/P3-Q 차단과 residual/raw 동등성 경고를 유지한다.


In [6]:
# 이 셀은 P3 표본과 OOF residual의 예측 성능을 확인한다.
p3_sample = pd.read_csv(PATHS["p3_sample"])
p3_q_gate = pd.read_csv(PATHS["p3_q_gate"])
p3_oof = pd.read_csv(PATHS["p3_oof"])
p3_test = pd.read_csv(PATHS["p3_test"])
p3_diag = pd.read_csv(PATHS["p3_diag"])

display(p3_sample)
display(p3_q_gate)
display(p3_oof)
display(p3_diag)

full_diag = p3_diag.loc[p3_diag["model_label"].eq("FULL")].iloc[0]
nopolicy_diag = p3_diag.loc[p3_diag["model_label"].eq("NOPOLICY")].iloc[0]
full_oof = p3_oof.loc[p3_oof["model_label"].eq("FULL")].iloc[0]
nopolicy_oof = p3_oof.loc[p3_oof["model_label"].eq("NOPOLICY")].iloc[0]

display(Markdown(
    f"""**P3 해석:** FULL residual coverage는 `{int(full_diag.coverage_n):,}`행, coverage rate는 `{full_diag.coverage_rate:.3f}`다.
FULL OOF 성능은 MAE `{full_oof.oof_mae:.3f}`, RMSE `{full_oof.oof_rmse:.3f}`, R² `{full_oof.oof_r2:.3f}`이다.
locked test R²는 `{full_oof.test_r2:.3f}`로 낮아, residual은 강한 예측모형의 순수 잔차라기보다 약한 baseline 이후 남은 학과·학교 이질성을 많이 포함한다.
raw A와 FULL residual의 상관은 `{full_diag.raw_residual_corr:.3f}`, residual/raw variance ratio는 `{full_diag.residual_to_raw_variance_ratio:.3f}`다.
NOPOLICY OOF R²는 `{nopolicy_oof.oof_r2:.3f}`이며 FULL보다 약간 높지만, P3 primary는 정책까지 조건화한 FULL이다."""
))


,sample_id,row_n,registry_row_n,school_n,registry_school_n,train_n,registry_train_n,validation_n,registry_validation_n,test_n,registry_test_n,target_non_null_n,row_id_hash
0,P3_STRUCTURE_READY,7592,7592,197,197,5534,5534,1168,1168,890,890,7592,2e3fb7557ee894b50542591376211ec84817501cd7b412...
1,P3_SELECTIVITY_READY,3119,3119,146,146,2293,2293,514,514,312,312,3119,1a0e36bb24d7e7f894897503bf0baf2c36f9b8710978b9...


,branch,status,reason
0,P3-Q,BLOCKED_FEATURE_CONTRACT,inherits P2-Q feature approval gate


,model_label,dev_row_n,test_row_n,oof_n,oof_mae,oof_rmse,oof_r2,oof_calibration_intercept,oof_calibration_slope,test_n,test_mae,test_rmse,test_r2,test_calibration_intercept,test_calibration_slope,devfit_rank,devfit_encoded_n
0,FULL,6702,890,6702,10.473046,14.053527,0.056153,12.081454,0.711055,890,10.432357,14.082241,-0.066762,25.082007,0.377768,38,38
1,NOPOLICY,6702,890,6702,10.375679,13.952432,0.069684,9.840046,0.764013,890,10.426699,14.085459,-0.067249,25.143242,0.376313,37,37


,model_label,coverage_n,coverage_rate,raw_residual_corr,residual_variance,raw_variance,residual_to_raw_variance_ratio,expected_mean,residual_mean
0,FULL,7592,1.0,0.926507,197.530286,209.282974,0.943843,41.788217,-0.123777
1,NOPOLICY,7592,1.0,0.926750,194.699208,209.282974,0.930316,41.825725,-0.161285


**P3 해석:** FULL residual coverage는 `7,592`행, coverage rate는 `1.000`다.
FULL OOF 성능은 MAE `10.473`, RMSE `14.054`, R² `0.056`이다.
locked test R²는 `-0.067`로 낮아, residual은 강한 예측모형의 순수 잔차라기보다 약한 baseline 이후 남은 학과·학교 이질성을 많이 포함한다.
raw A와 FULL residual의 상관은 `0.927`, residual/raw variance ratio는 `0.944`다.
NOPOLICY OOF R²는 `0.070`이며 FULL보다 약간 높지만, P3 primary는 정책까지 조건화한 FULL이다.

## V02. P3 Residual Model Diagnostic

- 시각화 목적: OOF residual이 무엇을 남기는지 observed/expected/residual/fold metric으로 분해한다.
- 사용할 데이터: `P3_STRUCTURE_GRADE_RESIDUAL_FULL.parquet`, `P3_OOF_PERFORMANCE.csv`, `P3_FULL_FOLD_METRICS.csv`.
- 필요한 전처리: residual parquet 샘플링, OOF/test R2 long-form 변환.
- 코드 셀 설계: observed-vs-expected scatter, residual histogram, OOF/test R2 bar, fold stability line.
- 그래프 해석 포인트: residual은 raw A와 강하게 상관된 구조 통제 후 잔여 grade signal이다.
- 학생이 자주 하는 오해: residual을 곧바로 인과적 shock 또는 독립 신호로 읽는 것.
- 체크포인트 질문: locked-test R2가 낮은데 residual을 어떻게 제한적으로 써야 하는가?

In [7]:
# V02. P3 residual 모델의 실제 구조를 expected-vs-observed, residual, fold metric으로 읽는다.
p3_full_residual = pd.read_parquet(PATHS["p3_full_residual"])
p3_fold_metrics_path = P3_ROOT / "qa/P3_FULL_FOLD_METRICS.csv"
p3_fold_metrics = pd.read_csv(p3_fold_metrics_path) if p3_fold_metrics_path.exists() else pd.DataFrame()

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
ax = axes[0, 0]
sample_for_scatter = p3_full_residual.sample(min(2500, len(p3_full_residual)), random_state=42)
ax.scatter(
    sample_for_scatter["expected_a_rate_full_pct"],
    sample_for_scatter["a_rate_pct"],
    s=8,
    alpha=0.35,
    color="#4C78A8",
)
low = min(sample_for_scatter["expected_a_rate_full_pct"].min(), sample_for_scatter["a_rate_pct"].min())
high = max(sample_for_scatter["expected_a_rate_full_pct"].max(), sample_for_scatter["a_rate_pct"].max())
ax.plot([low, high], [low, high], color="black", linewidth=1)
ax.set_title("Observed A-rate vs P3 expected A-rate")
ax.set_xlabel("expected A-rate from P3 FULL (%)")
ax.set_ylabel("observed A-rate (%)")

ax = axes[0, 1]
ax.hist(p3_full_residual["grade_residual_structure_full_oof_pp"], bins=45, color="#59A14F", alpha=0.85)
ax.axvline(0, color="black", linewidth=1)
ax.set_title("OOF residual distribution")
ax.set_xlabel("observed - expected A-rate (percentage points)")
ax.set_ylabel("department rows")

ax = axes[1, 0]
perf_plot = p3_oof.melt(
    id_vars="model_label",
    value_vars=["oof_r2", "test_r2"],
    var_name="metric",
    value_name="value",
)
for idx, metric in enumerate(["oof_r2", "test_r2"]):
    sub = perf_plot[perf_plot["metric"].eq(metric)]
    x = np.arange(len(sub)) + (idx - 0.5) * 0.32
    ax.bar(x, sub["value"], width=0.3, label=metric)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(np.arange(p3_oof["model_label"].nunique()))
ax.set_xticklabels(p3_oof["model_label"].unique())
ax.set_title("OOF vs locked-test R2")
ax.set_ylabel("R2")
ax.legend()

ax = axes[1, 1]
if len(p3_fold_metrics):
    ax.plot(p3_fold_metrics["fold"], p3_fold_metrics["mae"], marker="o", label="MAE")
    ax2 = ax.twinx()
    ax2.plot(p3_fold_metrics["fold"], p3_fold_metrics["r2"], marker="s", color="#F58518", label="R2")
    ax.set_xlabel("Group fold")
    ax.set_ylabel("MAE")
    ax2.set_ylabel("R2")
    ax.set_title("P3 FULL fold stability")
    lines_1, labels_1 = ax.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax.legend(lines_1 + lines_2, labels_1 + labels_2, loc="best")
else:
    ax.axis("off")
    ax.text(0.02, 0.95, "P3_FULL_FOLD_METRICS.csv not found", va="top")
fig.tight_layout()
path = save_visual_figure(
    fig,
    "P2_G6_1_V02_P3_RESIDUAL_DIAGNOSTIC.png",
    "V02_P3_RESIDUAL_DIAGNOSTIC",
    "P3 residual은 강한 예측모형의 잔차인가, 구조 통제 후 남은 grade signal인가?",
    "P3_STRUCTURE_GRADE_RESIDUAL_FULL.parquet, P3_OOF_PERFORMANCE.csv, P3_FULL_FOLD_METRICS.csv",
)
plt.show()
print(f"saved visual: {path.relative_to(ROOT)}")

display_reading_note(
    "V02 해석",
    f"FULL residual coverage는 {len(p3_full_residual):,}행이고 raw A-residual corr는 {full_diag.raw_residual_corr:.3f}로 높다.",
    "P3 expected A-rate가 A비율의 일부 구조를 설명하지만, residual이 raw A와 강하게 같이 움직인다.",
    "locked test R2가 낮으므로 residual을 순수한 인과적 shock처럼 읽으면 안 된다.",
    "P6에서는 residual topology를 보되 raw A와 거의 같은 축을 공유한다는 경고를 유지한다.",
)

saved visual: workbook/p2/P2_6/figures/P2_G6_1_V02_P3_RESIDUAL_DIAGNOSTIC.png


/tmp/ipykernel_183879/811936422.py:71: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**V02 해석**

- 관찰: FULL residual coverage는 7,592행이고 raw A-residual corr는 0.927로 높다.
- 원인: P3 expected A-rate가 A비율의 일부 구조를 설명하지만, residual이 raw A와 강하게 같이 움직인다.
- 제한: locked test R2가 낮으므로 residual을 순수한 인과적 shock처럼 읽으면 안 된다.
- 결론: P6에서는 residual topology를 보되 raw A와 거의 같은 축을 공유한다는 경고를 유지한다.


In [8]:
# 이 셀은 P4 outcome 표본과 모형 진단을 확인한다.
p4_sample = pd.read_csv(PATHS["p4_sample"])
p4_coef = pd.read_csv(PATHS["p4_coef"])
p4_iv = pd.read_csv(PATHS["p4_iv"])
p4_d = pd.read_csv(PATHS["p4_d"])
p4_test = pd.read_csv(PATHS["p4_test"])
p4_boot_ci = pd.read_csv(PATHS["p4_boot_ci"])
p4_diag = pd.read_csv(PATHS["p4_diag"])
p4_holm = pd.read_csv(PATHS["p4_holm"])
p4_equiv = pd.read_csv(PATHS["p4_equiv"])

display(p4_sample)
display(p4_diag)

sample_text = []
for _, row in p4_sample.iterrows():
    sample_text.append(
        f"`{row.sample_id}`: N={int(row.row_n):,}, school_n={int(row.school_n):,}, "
        f"train={int(row.train_n):,}, validation={int(row.validation_n):,}, test={int(row.test_n):,}"
    )
display(Markdown("**P4 표본:**\n\n" + "\n\n".join(sample_text)))


,sample_id,row_n,registry_row_n,school_n,registry_school_n,train_n,registry_train_n,validation_n,registry_validation_n,test_n,registry_test_n,target_non_null_n,row_id_hash
0,P4_E_STRUCTURE_READY,5600,5600,185,185,4080,4080,871,871,649,649,5600,3f7f76bc4e1a9680052b996b012021d36ccf3374fa182f...
1,P4_P_STRUCTURE_READY,5674,5674,194,194,4129,4129,884,884,661,661,5674,6625f4fcb391ac2e7d9a3001be72425a2dd4f0f6e03ef4...
2,P4_JOINT_STRUCTURE_READY,5600,5600,185,185,4080,4080,871,871,649,649,5600,3f7f76bc4e1a9680052b996b012021d36ccf3374fa182f...


,model_id,outcome,grade_signal,model_family,status,converged,warning,rank,diagnostic_flag
0,HEALTH_EMPLOYMENT_RAW_A_fractional_logit,HEALTH_EMPLOYMENT,RAW_A,fractional_logit,READY,True,NaN,37.0,NaN
1,HEALTH_EMPLOYMENT_RAW_A_ols_cluster,HEALTH_EMPLOYMENT,RAW_A,ols_cluster,READY,True,NaN,37.0,NaN
2,HEALTH_EMPLOYMENT_OOF_RESIDUAL_FULL_fractional...,HEALTH_EMPLOYMENT,OOF_RESIDUAL_FULL,fractional_logit,READY,True,NaN,37.0,NaN
3,HEALTH_EMPLOYMENT_OOF_RESIDUAL_FULL_ols_cluster,HEALTH_EMPLOYMENT,OOF_RESIDUAL_FULL,ols_cluster,READY,True,NaN,37.0,NaN
4,HEALTH_EMPLOYMENT_OOF_RESIDUAL_NOPOLICY_fracti...,HEALTH_EMPLOYMENT,OOF_RESIDUAL_NOPOLICY,fractional_logit,READY,True,NaN,37.0,NaN
5,HEALTH_EMPLOYMENT_OOF_RESIDUAL_NOPOLICY_ols_cl...,HEALTH_EMPLOYMENT,OOF_RESIDUAL_NOPOLICY,ols_cluster,READY,True,NaN,37.0,NaN
6,GRAD_SCHOOL_PROGRESSION_RAW_A_fractional_logit,GRAD_SCHOOL_PROGRESSION,RAW_A,fractional_logit,READY,True,NaN,39.0,NaN
7,GRAD_SCHOOL_PROGRESSION_RAW_A_ols_cluster,GRAD_SCHOOL_PROGRESSION,RAW_A,ols_cluster,READY,True,NaN,39.0,NaN
8,GRAD_SCHOOL_PROGRESSION_OOF_RESIDUAL_FULL_frac...,GRAD_SCHOOL_PROGRESSION,OOF_RESIDUAL_FULL,fractional_logit,READY,True,NaN,39.0,NaN
9,GRAD_SCHOOL_PROGRESSION_OOF_RESIDUAL_FULL_ols_...,GRAD_SCHOOL_PROGRESSION,OOF_RESIDUAL_FULL,ols_cluster,READY,True,NaN,39.0,NaN


**P4 표본:**

`P4_E_STRUCTURE_READY`: N=5,600, school_n=185, train=4,080, validation=871, test=649

`P4_P_STRUCTURE_READY`: N=5,674, school_n=194, train=4,129, validation=884, test=661

`P4_JOINT_STRUCTURE_READY`: N=5,600, school_n=185, train=4,080, validation=871, test=649

## V03. P4 Sample / Outcome Structure

- 시각화 목적: outcome model이 어떤 표본 분할과 결측 universe 위에서 비교되는지 확인한다.
- 사용할 데이터: `P4_SAMPLE_AUDIT.csv`, `P4_STRUCTURE_JOINT_FRAME.parquet`.
- 필요한 전처리: train/validation/test count와 outcome availability 집계.
- 코드 셀 설계: split stacked bar와 outcome/signal availability stacked bar.
- 그래프 해석 포인트: slope 비교는 표본 universe와 outcome coverage를 함께 읽어야 한다.
- 학생이 자주 하는 오해: 같은 학과 수로 모든 outcome이 동시에 추정됐다고 보는 것.
- 체크포인트 질문: 취업률과 진학률 비교에서 결측 구조는 어떤 제한을 주는가?

In [9]:
# V03. P4 outcome model이 어떤 표본과 split 위에서 돌아갔는지 시각화한다.
p4_joint = pd.read_parquet(P4_ROOT / "data/P4_STRUCTURE_JOINT_FRAME.parquet")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sample_cols = [c for c in ["train_n", "validation_n", "test_n"] if c in p4_sample.columns]
bottom = np.zeros(len(p4_sample))
x = np.arange(len(p4_sample))
for col, color in zip(sample_cols, ["#4C78A8", "#F58518", "#54A24B"]):
    axes[0].bar(x, p4_sample[col], bottom=bottom, label=col.replace("_n", ""), color=color)
    bottom += p4_sample[col].to_numpy()
axes[0].set_xticks(x)
axes[0].set_xticklabels(p4_sample["sample_id"], rotation=25, ha="right")
axes[0].set_ylabel("rows")
axes[0].set_title("P4 train/validation/test split by sample")
axes[0].legend()

coverage = pd.DataFrame(
    [
        {
            "outcome": "health_employment",
            "non_null_n": p4_joint["health_employment_rate_prop"].notna().sum(),
            "null_n": p4_joint["health_employment_rate_prop"].isna().sum(),
        },
        {
            "outcome": "graduate_progression",
            "non_null_n": p4_joint["graduate_school_progression_rate_prop"].notna().sum(),
            "null_n": p4_joint["graduate_school_progression_rate_prop"].isna().sum(),
        },
        {
            "outcome": "OOF residual",
            "non_null_n": p4_joint["grade_residual_structure_full_oof_10pp"].notna().sum(),
            "null_n": p4_joint["grade_residual_structure_full_oof_10pp"].isna().sum(),
        },
    ]
)
axes[1].bar(coverage["outcome"], coverage["non_null_n"], label="available", color="#72B7B2")
axes[1].bar(coverage["outcome"], coverage["null_n"], bottom=coverage["non_null_n"], label="missing", color="#BAB0AC")
axes[1].set_title("P4 outcome/signal availability")
axes[1].set_ylabel("rows in joint frame")
axes[1].tick_params(axis="x", rotation=20)
axes[1].legend()
fig.tight_layout()
path = save_visual_figure(
    fig,
    "P2_G6_1_V03_P4_SAMPLE_STRUCTURE.png",
    "V03_P4_SAMPLE_STRUCTURE",
    "P4 모형의 비교가 같은 표본 구조와 outcome availability 위에서 이루어졌는가?",
    "P4_SAMPLE_AUDIT.csv, P4_STRUCTURE_JOINT_FRAME.parquet",
)
plt.show()
print(f"saved visual: {path.relative_to(ROOT)}")

display_reading_note(
    "V03 해석",
    "P4는 outcome별 구조 표본과 joint frame의 결측/사용 가능 범위 위에서 읽어야 한다.",
    "취업률·대학원 진학률은 같은 학과 universe에서 항상 동시에 관측되는 변수가 아니다.",
    "표본 수 차이는 효과 크기 차이와 별개의 불확실성 원천이다.",
    "P6 결과 담론에는 slope뿐 아니라 표본 universe와 missing pattern을 함께 명시한다.",
)

saved visual: workbook/p2/P2_6/figures/P2_G6_1_V03_P4_SAMPLE_STRUCTURE.png


/tmp/ipykernel_183879/1542241967.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**V03 해석**

- 관찰: P4는 outcome별 구조 표본과 joint frame의 결측/사용 가능 범위 위에서 읽어야 한다.
- 원인: 취업률·대학원 진학률은 같은 학과 universe에서 항상 동시에 관측되는 변수가 아니다.
- 제한: 표본 수 차이는 효과 크기 차이와 별개의 불확실성 원천이다.
- 결론: P6 결과 담론에는 slope뿐 아니라 표본 universe와 missing pattern을 함께 명시한다.


In [10]:
# 이 셀은 fractional logit primary 결과를 percentage point 단위로 읽기 쉽게 변환한다.
primary_signals = ["RAW_A", "OOF_RESIDUAL_FULL"]
frac = p4_coef.loc[
    p4_coef["model_family"].eq("fractional_logit") & p4_coef["grade_signal"].isin(primary_signals)
].copy()
frac["ame_pp_10pp"] = frac["ame_10pp"] * 100
frac["iv_deviance_pp"] = frac["iv_deviance"] * 100
frac["iv_brier_pp"] = frac["iv_brier"] * 100
frac_display = frac[
    [
        "outcome",
        "grade_signal",
        "N",
        "school_n",
        "beta",
        "cluster_se",
        "ci_low",
        "ci_high",
        "p_value",
        "p_holm",
        "odds_ratio_10pp",
        "ame_pp_10pp",
        "iv_deviance",
        "iv_brier",
        "iv_mae",
        "test_deviance",
        "test_mae",
    ]
].sort_values(["outcome", "grade_signal"])
display(frac_display)

interpretation_lines = []
for _, row in frac_display.iterrows():
    outcome_kr = "건강보험 취업률" if row.outcome == "HEALTH_EMPLOYMENT" else "대학원 진학률"
    signal_kr = "raw A" if row.grade_signal == "RAW_A" else "P3 FULL OOF residual"
    interpretation_lines.append(
        f"- {outcome_kr} / {signal_kr}: A signal 10%p 증가와 연결된 평균 예측확률 변화 AME는 "
        f"{row.ame_pp_10pp:+.3f}%p, OR은 {row.odds_ratio_10pp:.3f}, Holm p는 {row.p_holm:.3g}이다."
    )
display(Markdown("**P4 primary slope 해석:**\n\n" + "\n".join(interpretation_lines)))


,outcome,grade_signal,N,school_n,beta,cluster_se,ci_low,ci_high,p_value,p_holm,odds_ratio_10pp,ame_pp_10pp,iv_deviance,iv_brier,iv_mae,test_deviance,test_mae
8,GRAD_SCHOOL_PROGRESSION,OOF_RESIDUAL_FULL,5013,165,0.220272,0.031812,0.157922,0.282622,4.383423e-12,2.630054e-11,1.246416,1.657726,0.005411,0.000669,0.002103,0.542177,0.093240
6,GRAD_SCHOOL_PROGRESSION,RAW_A,5013,165,0.228680,0.033250,0.163513,0.293848,6.082297e-12,3.041148e-11,1.256940,1.725986,0.005394,0.000666,0.002048,0.566943,0.101346
2,HEALTH_EMPLOYMENT,OOF_RESIDUAL_FULL,4951,158,0.026359,0.011701,0.003426,0.049293,2.427163e-02,7.281490e-02,1.026710,0.635722,0.000262,0.000068,0.000173,1.360693,0.125346
0,HEALTH_EMPLOYMENT,RAW_A,4951,158,0.025553,0.012006,0.002021,0.049084,3.331334e-02,7.281490e-02,1.025882,0.616300,0.000225,0.000058,0.000158,1.362097,0.125970


**P4 primary slope 해석:**

- 대학원 진학률 / P3 FULL OOF residual: A signal 10%p 증가와 연결된 평균 예측확률 변화 AME는 +1.658%p, OR은 1.246, Holm p는 2.63e-11이다.
- 대학원 진학률 / raw A: A signal 10%p 증가와 연결된 평균 예측확률 변화 AME는 +1.726%p, OR은 1.257, Holm p는 3.04e-11이다.
- 건강보험 취업률 / P3 FULL OOF residual: A signal 10%p 증가와 연결된 평균 예측확률 변화 AME는 +0.636%p, OR은 1.027, Holm p는 0.0728이다.
- 건강보험 취업률 / raw A: A signal 10%p 증가와 연결된 평균 예측확률 변화 AME는 +0.616%p, OR은 1.026, Holm p는 0.0728이다.

## V04. P4 AME Forest Plot

- 시각화 목적: RAW_A와 OOF residual의 outcome별 평균한계효과를 CI와 함께 비교한다.
- 사용할 데이터: `P4_COEFFICIENT_RESULTS.csv`, `P4_BOOTSTRAP_CI.csv`.
- 필요한 전처리: AME를 percentage point로 변환하고 bootstrap CI를 merge한다.
- 코드 셀 설계: outcome/signal별 horizontal forest plot.
- 그래프 해석 포인트: 대학원 진학률 slope가 취업률 slope보다 크고, RAW_A/OOF residual은 거의 겹친다.
- 학생이 자주 하는 오해: p-value만 보고 effect size와 외부예측 개선을 생략하는 것.
- 체크포인트 질문: 같은 AME라도 outcome baseline과 표본 차이에 따라 해석이 어떻게 달라지는가?

In [11]:
# V04. P4 fractional logit primary slope를 bootstrap AME CI와 함께 비교한다.
ame_boot = p4_boot_ci.loc[p4_boot_ci["metric"].eq("ame")].copy()
ame_boot["ame_mid_pp"] = ame_boot["median"] * 100
ame_boot["ci_low_pp"] = ame_boot["ci_low"] * 100
ame_boot["ci_high_pp"] = ame_boot["ci_high"] * 100
ame_plot = frac_display.merge(
    ame_boot[["outcome", "grade_signal", "ci_low_pp", "ci_high_pp", "ame_mid_pp"]],
    on=["outcome", "grade_signal"],
    how="left",
)
ame_plot["row_label"] = ame_plot["outcome"].map(
    {"HEALTH_EMPLOYMENT": "Health employment", "GRAD_SCHOOL_PROGRESSION": "Grad progression"}
) + " / " + ame_plot["grade_signal"].map({"RAW_A": "RAW_A", "OOF_RESIDUAL_FULL": "OOF residual"})
ame_plot = ame_plot.sort_values(["outcome", "grade_signal"]).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9.2, 4.8))
y = np.arange(len(ame_plot))
xerr = np.vstack(
    [
        ame_plot["ame_pp_10pp"] - ame_plot["ci_low_pp"],
        ame_plot["ci_high_pp"] - ame_plot["ame_pp_10pp"],
    ]
)
colors = ["#4C78A8" if signal == "RAW_A" else "#F58518" for signal in ame_plot["grade_signal"]]
ax.errorbar(ame_plot["ame_pp_10pp"], y, xerr=xerr, fmt="none", ecolor="#6B6B6B", elinewidth=1.5, capsize=4)
ax.scatter(ame_plot["ame_pp_10pp"], y, s=80, color=colors, zorder=3)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_yticks(y)
ax.set_yticklabels(ame_plot["row_label"])
ax.set_xlabel("AME per 10pp grade signal (percentage points)")
ax.set_title("P4 primary signal size with generated-regressor bootstrap CI")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
path = save_visual_figure(
    fig,
    "P2_G6_1_V04_P4_AME_FOREST.png",
    "V04_P4_AME_FOREST",
    "RAW_A와 OOF residual은 취업률/대학원 진학률에 어느 정도의 추가 신호를 주는가?",
    "P4_COEFFICIENT_RESULTS.csv, P4_BOOTSTRAP_CI.csv",
)
plt.show()
print(f"saved visual: {path.relative_to(ROOT)}")

display_reading_note(
    "V04 해석",
    "대학원 진학률의 AME가 건강보험 취업률보다 더 크게 보이며, RAW_A와 OOF residual의 위치가 거의 겹친다.",
    "같은 P4 controls 안에서는 residual과 raw A가 같은 added-information 방향을 span한다.",
    "bootstrap CI는 generated-regressor approximation 경고가 있으므로 정밀한 확증보다 방향성 판단에 둔다.",
    "P6 담론은 'grade signal은 취업보다 대학원 진학과 더 정렬'이라는 제한적 결론으로 유지한다.",
)

saved visual: workbook/p2/P2_6/figures/P2_G6_1_V04_P4_AME_FOREST.png


/tmp/ipykernel_183879/1758874010.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**V04 해석**

- 관찰: 대학원 진학률의 AME가 건강보험 취업률보다 더 크게 보이며, RAW_A와 OOF residual의 위치가 거의 겹친다.
- 원인: 같은 P4 controls 안에서는 residual과 raw A가 같은 added-information 방향을 span한다.
- 제한: bootstrap CI는 generated-regressor approximation 경고가 있으므로 정밀한 확증보다 방향성 판단에 둔다.
- 결론: P6 담론은 'grade signal은 취업보다 대학원 진학과 더 정렬'이라는 제한적 결론으로 유지한다.


In [12]:
# 이 셀은 대학원 진학 slope와 건강보험 취업 slope의 직접 차이 D를 확인한다.
p4_d_display = p4_d.copy()
for col in ["GRAD_SCHOOL_PROGRESSION", "HEALTH_EMPLOYMENT", "D_progression_minus_employment"]:
    p4_d_display[col + "_pp"] = p4_d_display[col] * 100
display(p4_d_display)

boot_display = p4_boot_ci.copy()
for col in ["ci_low", "ci_high", "median"]:
    boot_display[col + "_pp"] = boot_display[col] * 100
display(boot_display)

d_raw = p4_d.loc[p4_d["grade_signal"].eq("RAW_A"), "D_progression_minus_employment"].iloc[0] * 100
d_resid = p4_d.loc[p4_d["grade_signal"].eq("OOF_RESIDUAL_FULL"), "D_progression_minus_employment"].iloc[0] * 100
d_boot = boot_display.loc[
    boot_display["outcome"].eq("PROGRESSION_MINUS_EMPLOYMENT")
    & boot_display["grade_signal"].eq("OOF_RESIDUAL_FULL")
].iloc[0]

display(Markdown(
    f"""**D 해석:** RAW_A 기준 대학원 진학 AME - 건강보험 취업 AME는 `{d_raw:+.3f}%p`이고,
OOF_RESIDUAL_FULL 기준은 `{d_resid:+.3f}%p`다.
generated-regressor school bootstrap에서 residual D의 95% percentile CI는
`[{d_boot.ci_low_pp:+.3f}, {d_boot.ci_high_pp:+.3f}]%p`, median은 `{d_boot.median_pp:+.3f}%p`다.
이는 P4 구조분기에서 grade signal이 취업률보다 대학원 진학률과 더 크게 정렬된다는 탐색적 근거다."""
))


,grade_signal,GRAD_SCHOOL_PROGRESSION,HEALTH_EMPLOYMENT,D_progression_minus_employment,GRAD_SCHOOL_PROGRESSION_pp,HEALTH_EMPLOYMENT_pp,D_progression_minus_employment_pp
0,OOF_RESIDUAL_FULL,0.016577,0.006357,0.010220,1.657726,0.635722,1.022005
1,OOF_RESIDUAL_NOPOLICY,0.016645,0.006200,0.010445,1.664520,0.620001,1.044519
2,RAW_A,0.017260,0.006163,0.011097,1.725986,0.616300,1.109686


,outcome,grade_signal,metric,successful_n,ci_low,ci_high,median,ci_low_pp,ci_high_pp,median_pp
0,GRAD_SCHOOL_PROGRESSION,OOF_RESIDUAL_FULL,ame,500,0.010073,0.024457,0.016740,1.007348,2.445664,1.673958
1,GRAD_SCHOOL_PROGRESSION,RAW_A,ame,500,0.010073,0.024457,0.016740,1.007348,2.445664,1.673958
2,HEALTH_EMPLOYMENT,OOF_RESIDUAL_FULL,ame,500,-0.000431,0.011780,0.005860,-0.043081,1.178028,0.586003
3,HEALTH_EMPLOYMENT,RAW_A,ame,500,-0.000431,0.011780,0.005860,-0.043081,1.178028,0.586003
4,PROGRESSION_MINUS_EMPLOYMENT,OOF_RESIDUAL_FULL,D,500,0.002916,0.018298,0.010658,0.291583,1.829812,1.065826
5,PROGRESSION_MINUS_EMPLOYMENT,RAW_A,D,500,0.002916,0.018298,0.010658,0.291582,1.829812,1.065826


**D 해석:** RAW_A 기준 대학원 진학 AME - 건강보험 취업 AME는 `+1.110%p`이고,
OOF_RESIDUAL_FULL 기준은 `+1.022%p`다.
generated-regressor school bootstrap에서 residual D의 95% percentile CI는
`[+0.292, +1.830]%p`, median은 `+1.066%p`다.
이는 P4 구조분기에서 grade signal이 취업률보다 대학원 진학률과 더 크게 정렬된다는 탐색적 근거다.

## V05. Employment vs Progression Difference D

- 시각화 목적: grade signal이 취업률보다 대학원 진학률에 더 크게 정렬되는지 직접 차이로 본다.
- 사용할 데이터: `P4_EMPLOYMENT_PROGRESSION_DIFFERENCE.csv`, `P4_BOOTSTRAP_CI.csv`.
- 필요한 전처리: D와 bootstrap CI를 percentage point 단위로 변환한다.
- 코드 셀 설계: signal별 D point + CI plot.
- 그래프 해석 포인트: D가 양수면 progression slope가 employment slope보다 크다.
- 학생이 자주 하는 오해: D를 두 독립 모형의 단순 눈대중 차이로 처리하는 것.
- 체크포인트 질문: D 결과가 P6의 결과론적 담론을 어떻게 바꾸는가?

In [13]:
# V05. Employment-vs-progression 차이 D를 별도 결과물로 분리한다.
d_plot = p4_d.copy()
d_plot["D_pp"] = d_plot["D_progression_minus_employment"] * 100
d_ci = p4_boot_ci.loc[p4_boot_ci["outcome"].eq("PROGRESSION_MINUS_EMPLOYMENT")].copy()
d_ci["ci_low_pp"] = d_ci["ci_low"] * 100
d_ci["ci_high_pp"] = d_ci["ci_high"] * 100
d_ci["median_pp"] = d_ci["median"] * 100
d_plot = d_plot.merge(d_ci[["grade_signal", "ci_low_pp", "ci_high_pp", "median_pp"]], on="grade_signal", how="left")
d_plot = d_plot.sort_values("grade_signal").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8.5, 4.5))
y = np.arange(len(d_plot))
xerr = np.vstack([d_plot["D_pp"] - d_plot["ci_low_pp"], d_plot["ci_high_pp"] - d_plot["D_pp"]])
colors = ["#4C78A8" if signal == "RAW_A" else "#F58518" for signal in d_plot["grade_signal"]]
ax.errorbar(d_plot["D_pp"], y, xerr=xerr, fmt="none", ecolor="#6B6B6B", elinewidth=1.5, capsize=4)
ax.scatter(d_plot["D_pp"], y, s=90, color=colors, zorder=3)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_yticks(y)
ax.set_yticklabels(d_plot["grade_signal"])
ax.set_xlabel("Progression AME - Employment AME (percentage points)")
ax.set_title("Does grade signal align more with progression than employment?")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
path = save_visual_figure(
    fig,
    "P2_G6_1_V05_D_BOOTSTRAP.png",
    "V05_D_BOOTSTRAP",
    "grade signal의 추가 효과가 취업률보다 대학원 진학률에서 더 큰가?",
    "P4_EMPLOYMENT_PROGRESSION_DIFFERENCE.csv, P4_BOOTSTRAP_CI.csv",
)
plt.show()
print(f"saved visual: {path.relative_to(ROOT)}")

display_reading_note(
    "V05 해석",
    f"OOF residual 기준 D는 {d_resid:+.3f}%p이고 bootstrap CI가 양의 영역을 중심으로 놓인다.",
    "D는 두 outcome slope의 차이이므로 단일 outcome의 유의성과 다르게 해석해야 한다.",
    "대학원 진학률은 진학 의사·전공별 구조·학교 prestige를 함께 반영할 수 있어 causal label로 읽지 않는다.",
    "P6에서는 '취업 성과 개선'보다 '진학 선택/경로와의 정렬'이라는 결과 담론이 더 안전하다.",
)

saved visual: workbook/p2/P2_6/figures/P2_G6_1_V05_D_BOOTSTRAP.png


/tmp/ipykernel_183879/1402471092.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**V05 해석**

- 관찰: OOF residual 기준 D는 +1.022%p이고 bootstrap CI가 양의 영역을 중심으로 놓인다.
- 원인: D는 두 outcome slope의 차이이므로 단일 outcome의 유의성과 다르게 해석해야 한다.
- 제한: 대학원 진학률은 진학 의사·전공별 구조·학교 prestige를 함께 반영할 수 있어 causal label로 읽지 않는다.
- 결론: P6에서는 '취업 성과 개선'보다 '진학 선택/경로와의 정렬'이라는 결과 담론이 더 안전하다.


In [14]:
# 이 셀은 locked test에서 base 대비 signal 추가가 실제 외부 예측을 개선했는지 확인한다.
test_frac = p4_test.loc[p4_test["model_family"].eq("fractional_logit")].copy()
test_frac["test_deviance_improvement"] = test_frac["base_test_deviance"] - test_frac["test_deviance"]
test_frac["test_brier_improvement"] = test_frac["base_test_brier"] - test_frac["test_brier"]
test_frac["test_mae_improvement"] = test_frac["base_test_mae"] - test_frac["test_mae"]
display(test_frac.sort_values(["outcome", "grade_signal"]))

emp_resid = test_frac.loc[
    test_frac["outcome"].eq("HEALTH_EMPLOYMENT") & test_frac["grade_signal"].eq("OOF_RESIDUAL_FULL")
].iloc[0]
prog_resid = test_frac.loc[
    test_frac["outcome"].eq("GRAD_SCHOOL_PROGRESSION") & test_frac["grade_signal"].eq("OOF_RESIDUAL_FULL")
].iloc[0]
display(Markdown(
    f"""**Locked test 해석:** 건강보험 취업률에서 OOF residual의 deviance 개선은
`{emp_resid.test_deviance_improvement:+.6f}`로 거의 없고, MAE 개선은 `{emp_resid.test_mae_improvement:+.6f}`다.
대학원 진학률에서 OOF residual의 deviance 개선은 `{prog_resid.test_deviance_improvement:+.6f}`,
MAE 개선은 `{prog_resid.test_mae_improvement:+.6f}`로 더 뚜렷하다.
test 결과는 사양 선택에 재사용하지 않고 최종 잠금 평가로만 해석한다."""
))


,outcome,grade_signal,model_family,test_n,test_deviance,test_brier,test_mae,base_test_deviance,base_test_brier,base_test_mae,test_deviance_improvement,test_brier_improvement,test_mae_improvement
8,GRAD_SCHOOL_PROGRESSION,OOF_RESIDUAL_FULL,fractional_logit,661,0.542177,0.021416,0.093240,0.571045,0.028035,0.103206,0.028868,0.006619,0.009966
10,GRAD_SCHOOL_PROGRESSION,OOF_RESIDUAL_NOPOLICY,fractional_logit,661,0.542244,0.021445,0.093279,0.571045,0.028035,0.103206,0.028801,0.006590,0.009926
6,GRAD_SCHOOL_PROGRESSION,RAW_A,fractional_logit,661,0.566943,0.027118,0.101346,0.571045,0.028035,0.103206,0.004102,0.000917,0.001859
2,HEALTH_EMPLOYMENT,OOF_RESIDUAL_FULL,fractional_logit,649,1.360693,0.026899,0.125346,1.360279,0.026793,0.125622,-0.000414,-0.000107,0.000277
4,HEALTH_EMPLOYMENT,OOF_RESIDUAL_NOPOLICY,fractional_logit,649,1.360661,0.026892,0.125341,1.360279,0.026793,0.125622,-0.000382,-0.000099,0.000282
0,HEALTH_EMPLOYMENT,RAW_A,fractional_logit,649,1.362097,0.027230,0.125970,1.360279,0.026793,0.125622,-0.001818,-0.000437,-0.000348


**Locked test 해석:** 건강보험 취업률에서 OOF residual의 deviance 개선은
`-0.000414`로 거의 없고, MAE 개선은 `+0.000277`다.
대학원 진학률에서 OOF residual의 deviance 개선은 `+0.028868`,
MAE 개선은 `+0.009966`로 더 뚜렷하다.
test 결과는 사양 선택에 재사용하지 않고 최종 잠금 평가로만 해석한다.

## V06. Locked-Test Improvement

- 시각화 목적: signal 추가가 base 대비 test 성능을 실제로 개선했는지 metric별로 분리한다.
- 사용할 데이터: `P4_LOCKED_TEST_METRICS.csv`.
- 필요한 전처리: base metric - signal metric을 improvement로 계산한다.
- 코드 셀 설계: deviance, Brier, MAE improvement bar.
- 그래프 해석 포인트: slope 유의성과 locked-test 개선은 같은 질문이 아니다.
- 학생이 자주 하는 오해: coefficient가 유의하면 test metric도 반드시 좋아진다고 믿는 것.
- 체크포인트 질문: test 결과를 보고 사양을 다시 고르면 어떤 문제가 생기는가?

In [15]:
# V06. Locked test에서 signal 추가가 base 대비 실제로 좋아졌는지 metric별로 본다.
imp_plot = test_frac.loc[test_frac["model_family"].eq("fractional_logit")].copy()
imp_long = imp_plot.melt(
    id_vars=["outcome", "grade_signal"],
    value_vars=["test_deviance_improvement", "test_brier_improvement", "test_mae_improvement"],
    var_name="metric",
    value_name="improvement",
)
imp_long["panel"] = imp_long["outcome"].map(
    {"HEALTH_EMPLOYMENT": "Health employment", "GRAD_SCHOOL_PROGRESSION": "Grad progression"}
).fillna(imp_long["outcome"].astype(str))
imp_long["signal_label"] = imp_long["grade_signal"].map(
    {"RAW_A": "RAW_A", "OOF_RESIDUAL_FULL": "OOF residual"}
).fillna(imp_long["grade_signal"].astype(str))

fig, axes = plt.subplots(1, 3, figsize=(14, 4.6), sharey=False)
for ax, metric in zip(axes, ["test_deviance_improvement", "test_brier_improvement", "test_mae_improvement"]):
    sub = imp_long[imp_long["metric"].eq(metric)].copy()
    sub["label"] = sub["panel"].astype(str) + "\n" + sub["signal_label"].astype(str)
    colors = ["#59A14F" if v > 0 else "#E15759" for v in sub["improvement"]]
    ax.bar(sub["label"], sub["improvement"], color=colors)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(metric.replace("test_", "").replace("_improvement", " improvement"))
    ax.tick_params(axis="x", rotation=0)
    ax.grid(axis="y", alpha=0.25)
fig.suptitle("Locked-test improvement: positive means signal beats base", y=1.03)
fig.tight_layout()
path = save_visual_figure(
    fig,
    "P2_G6_1_V06_LOCKED_TEST_IMPROVEMENT.png",
    "V06_LOCKED_TEST_IMPROVEMENT",
    "P4 signal 추가가 locked test에서 base 대비 외부 예측을 개선했는가?",
    "P4_LOCKED_TEST_METRICS.csv",
)
plt.show()
print(f"saved visual: {path.relative_to(ROOT)}")

display_reading_note(
    "V06 해석",
    "건강보험 취업률에서는 improvement가 약하거나 음수이고, 대학원 진학률에서는 더 안정적인 개선이 보인다.",
    "P4의 slope 유의성과 locked-test 예측 개선은 같은 질문이 아니다.",
    "test 결과를 사양 선택에 되먹이면 잠금 평가의 의미가 사라진다.",
    "P6 문장에서는 effect-size와 out-of-sample 개선을 분리해서 보고한다.",
)

saved visual: workbook/p2/P2_6/figures/P2_G6_1_V06_LOCKED_TEST_IMPROVEMENT.png


/tmp/ipykernel_183879/3455391063.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**V06 해석**

- 관찰: 건강보험 취업률에서는 improvement가 약하거나 음수이고, 대학원 진학률에서는 더 안정적인 개선이 보인다.
- 원인: P4의 slope 유의성과 locked-test 예측 개선은 같은 질문이 아니다.
- 제한: test 결과를 사양 선택에 되먹이면 잠금 평가의 의미가 사라진다.
- 결론: P6 문장에서는 effect-size와 out-of-sample 개선을 분리해서 보고한다.


In [16]:
# 이 셀은 RAW_A와 OOF residual이 선형 P4 설계에서 사실상 같은 추가정보 차원을 제공하는지 감사한다.
display(p4_equiv)
display(p4_holm)

max_equiv_delta = p4_equiv["max_abs_raw_minus_residual"].max()
approx_warning = p4_diag.loc[p4_diag["diagnostic_flag"].eq("BOOTSTRAP_APPROXIMATION_WARNING")]

warning_lines = [
    f"RAW_A와 OOF_RESIDUAL_FULL의 end-to-end/bootstrapped 차이 최대값은 {max_equiv_delta:.3e} 수준이다.",
    "이는 residual = raw A - P3 expected A이고 P4에 같은 구조 통제가 들어갈 때 signal이 같은 1차원 정보를 span하기 때문이다.",
    "따라서 P5/P6에서 residual을 raw A와 완전히 다른 신호처럼 과장하지 않는다.",
]
if len(approx_warning):
    warning_lines.append(str(approx_warning.iloc[0]["warning"]))
display(Markdown("**동등성·bootstrap 경고:**\n\n" + "\n\n".join(f"- {x}" for x in warning_lines)))


,source,metric,max_abs_raw_minus_residual,allclose,reason
0,end_to_end_cv,iv_deviance,2.282073e-10,False,Residual equals raw A minus fitted P3 controls...
1,end_to_end_cv,iv_brier,3.377730e-11,True,Residual equals raw A minus fitted P3 controls...
2,end_to_end_cv,iv_mae,8.366750e-11,True,Residual equals raw A minus fitted P3 controls...
3,bootstrap_fast_generated_regressor,ame,7.581795e-10,True,Residual equals raw A minus fitted P3 controls...


,outcome,grade_signal,model_family,p_value,p_holm
0,HEALTH_EMPLOYMENT,RAW_A,fractional_logit,3.331334e-02,7.281490e-02
1,HEALTH_EMPLOYMENT,RAW_A,ols_cluster,3.328883e-02,NaN
2,HEALTH_EMPLOYMENT,OOF_RESIDUAL_FULL,fractional_logit,2.427163e-02,7.281490e-02
3,HEALTH_EMPLOYMENT,OOF_RESIDUAL_FULL,ols_cluster,2.446721e-02,NaN
4,HEALTH_EMPLOYMENT,OOF_RESIDUAL_NOPOLICY,fractional_logit,2.866542e-02,7.281490e-02
5,HEALTH_EMPLOYMENT,OOF_RESIDUAL_NOPOLICY,ols_cluster,2.886585e-02,NaN
6,GRAD_SCHOOL_PROGRESSION,RAW_A,fractional_logit,6.082297e-12,3.041148e-11
7,GRAD_SCHOOL_PROGRESSION,RAW_A,ols_cluster,1.405789e-06,NaN
8,GRAD_SCHOOL_PROGRESSION,OOF_RESIDUAL_FULL,fractional_logit,4.383423e-12,2.630054e-11
9,GRAD_SCHOOL_PROGRESSION,OOF_RESIDUAL_FULL,ols_cluster,9.229524e-07,NaN


**동등성·bootstrap 경고:**

- RAW_A와 OOF_RESIDUAL_FULL의 end-to-end/bootstrapped 차이 최대값은 7.582e-10 수준이다.

- 이는 residual = raw A - P3 expected A이고 P4에 같은 구조 통제가 들어갈 때 signal이 같은 1차원 정보를 span하기 때문이다.

- 따라서 P5/P6에서 residual을 raw A와 완전히 다른 신호처럼 과장하지 않는다.

- Bootstrap regenerates P3 residual inside each replicate but uses fixed development encoder and numpy least-squares for speed.

## V07. RAW_A / OOF Residual Equivalence Audit

- 시각화 목적: 현재 P4 선형 설계에서 RAW_A와 OOF residual이 거의 같은 추가정보 축인지 확인한다.
- 사용할 데이터: `P4_WITHIN_RAW_EQUIVALENCE_AUDIT.csv`, P3 residual parquet.
- 필요한 전처리: max delta log scale 변환과 raw/residual scatter 샘플링.
- 코드 셀 설계: metric delta bar와 raw A vs residual scatter.
- 그래프 해석 포인트: residual만의 별도 효과라고 과장하지 않는다.
- 학생이 자주 하는 오해: residual이라는 이름 때문에 raw A와 독립적인 신호라고 믿는 것.
- 체크포인트 질문: 같은 controls를 넣으면 왜 raw A와 residual이 같은 축을 공유하는가?

In [17]:
# V07. RAW_A와 OOF residual이 왜 거의 같은 추가정보 축인지 시각적으로 확인한다.
equiv_plot = p4_equiv.copy()
equiv_plot["log10_delta"] = np.log10(equiv_plot["max_abs_raw_minus_residual"].clip(lower=1e-15))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
axes[0].bar(equiv_plot["metric"], equiv_plot["log10_delta"], color="#B279A2")
axes[0].set_title("RAW_A vs OOF residual metric delta")
axes[0].set_ylabel("log10(max absolute delta)")
axes[0].tick_params(axis="x", rotation=20)
axes[0].grid(axis="y", alpha=0.25)

scatter_df = p3_full_residual.sample(min(3000, len(p3_full_residual)), random_state=7)
axes[1].scatter(
    scatter_df["a_rate_pct"],
    scatter_df["grade_residual_structure_full_oof_pp"],
    s=8,
    alpha=0.35,
    color="#4C78A8",
)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title(f"Raw A vs residual, corr={full_diag.raw_residual_corr:.3f}")
axes[1].set_xlabel("Raw A-rate (%)")
axes[1].set_ylabel("P3 FULL residual (pp)")
fig.tight_layout()
path = save_visual_figure(
    fig,
    "P2_G6_1_V07_RAW_RESID_EQUIVALENCE.png",
    "V07_RAW_RESID_EQUIVALENCE",
    "현재 P4 설계에서 RAW_A와 OOF residual을 독립적인 두 신호처럼 해석해도 되는가?",
    "P4_WITHIN_RAW_EQUIVALENCE_AUDIT.csv, P3_STRUCTURE_GRADE_RESIDUAL_FULL.parquet",
)
plt.show()
print(f"saved visual: {path.relative_to(ROOT)}")

display_reading_note(
    "V07 해석",
    "end-to-end delta는 극소값이고 raw A-residual 상관은 매우 높다.",
    "residual = raw A - P3 expected A이며, P4에 같은 구조 통제를 넣으면 같은 1차원 추가정보를 공유한다.",
    "이 동등성은 현재 선형 P4 설계 안에서의 성질이며 비선형/상호작용 설계까지 자동 확장되지 않는다.",
    "P6 residual topology는 residual 고유성보다 '어떤 구조에서 raw A 축이 남는가'를 묻는 방향으로 설계한다.",
)

saved visual: workbook/p2/P2_6/figures/P2_G6_1_V07_RAW_RESID_EQUIVALENCE.png


/tmp/ipykernel_183879/826990777.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**V07 해석**

- 관찰: end-to-end delta는 극소값이고 raw A-residual 상관은 매우 높다.
- 원인: residual = raw A - P3 expected A이며, P4에 같은 구조 통제를 넣으면 같은 1차원 추가정보를 공유한다.
- 제한: 이 동등성은 현재 선형 P4 설계 안에서의 성질이며 비선형/상호작용 설계까지 자동 확장되지 않는다.
- 결론: P6 residual topology는 residual 고유성보다 '어떤 구조에서 raw A 축이 남는가'를 묻는 방향으로 설계한다.


In [18]:
# 이 셀은 P6 및 다음 단계 상태를 명시적으로 분리한다.
p6_status = {
    "P6_INPUT_STATUS": "READY",
    "P6_P3_RESIDUAL_HANDOFF_STATUS": "READY",
    "P6_P4_CONFIRMATORY_STATUS": "READY_WITH_WARNINGS",
    "P6_P5_RESIDUAL_RERUN_STATUS": "READY",
    "P6_RESIDUAL_TOPOLOGY_STATUS": "READY_WITH_WARNINGS",
    "P6_Q_BRANCH_STATUS": "BLOCKED_FEATURE_CONTRACT",
    "P6_OVERALL_STATUS": "READY_WITH_WARNINGS",
}
df_p6_status = pd.DataFrame([{"status_key": k, "status": v} for k, v in p6_status.items()])
display(df_p6_status)

p5_handoff_display = pd.DataFrame(
    [
        {"key": k, "value": json.dumps(v, ensure_ascii=False) if isinstance(v, list) else v}
        for k, v in p4_to_p5.items()
    ]
)
display(p5_handoff_display)

display(Markdown(
    """**다음 단계 판정:** P4→P5 residual handoff가 존재하므로 P5 strict v2는 `RAW_A`와 `OOF_RESIDUAL_FULL`을 다시 비교할 수 있다.
다만 P6 residual topology에서는 raw A-residual 상관이 매우 높고 P4 선형 설계에서 두 signal이 사실상 같은 added-information 차원을 제공한다는 경고를 유지해야 한다.
P2-Q/P3-Q는 입결 feature approval 문제로 계속 차단 상태다."""
))


,status_key,status
0,P6_INPUT_STATUS,READY
1,P6_P3_RESIDUAL_HANDOFF_STATUS,READY
2,P6_P4_CONFIRMATORY_STATUS,READY_WITH_WARNINGS
3,P6_P5_RESIDUAL_RERUN_STATUS,READY
4,P6_RESIDUAL_TOPOLOGY_STATUS,READY_WITH_WARNINGS
5,P6_Q_BRANCH_STATUS,BLOCKED_FEATURE_CONTRACT
6,P6_OVERALL_STATUS,READY_WITH_WARNINGS


,key,value
0,created_at,2026-07-13T07:42:52.541358+00:00
1,p3_full_residual_path,workbook/p2/p2_4/p3_grade_residual_v1_strict/d...
2,p3_full_residual_sha256,d8decd39dca42ccd0dc194fa3813ba0036541098eea08d...
3,p3_nopolicy_residual_path,workbook/p2/p2_4/p3_grade_residual_v1_strict/d...
4,p3_nopolicy_residual_sha256,dc5675d941b17108997e63adb550f6f223911892085cf4...
5,recommended_p5_signals,"[""RAW_A"", ""OOF_RESIDUAL_FULL""]"


**다음 단계 판정:** P4→P5 residual handoff가 존재하므로 P5 strict v2는 `RAW_A`와 `OOF_RESIDUAL_FULL`을 다시 비교할 수 있다.
다만 P6 residual topology에서는 raw A-residual 상관이 매우 높고 P4 선형 설계에서 두 signal이 사실상 같은 added-information 차원을 제공한다는 경고를 유지해야 한다.
P2-Q/P3-Q는 입결 feature approval 문제로 계속 차단 상태다.

## V08. P6 Decision Dashboard

- 시각화 목적: P6로 넘길 준비 상태, 결과론적 claim, 유지할 경고를 한 장으로 고정한다.
- 사용할 데이터: P3/P4 summary objects와 `p6_status`.
- 필요한 전처리: question/finding/evidence/risk/next_action matrix 구성.
- 코드 셀 설계: decision matrix CSV 저장과 horizontal dashboard.
- 그래프 해석 포인트: READY와 BLOCKED가 같은 프로젝트 안에 공존할 수 있다.
- 학생이 자주 하는 오해: 전체 status 하나로 모든 branch가 승인됐다고 보는 것.
- 체크포인트 질문: P6가 열렸다는 말과 Q branch가 차단됐다는 말은 어떻게 동시에 참인가?

In [19]:
# V08. P6로 넘길 결과 담론을 구조화된 decision matrix로 고정한다.
decision_matrix = pd.DataFrame(
    [
        {
            "question": "P3 residual handoff",
            "finding": "READY",
            "evidence": f"coverage={int(full_diag.coverage_n):,}, coverage_rate={full_diag.coverage_rate:.3f}",
            "risk": "locked-test R2 is low; residual is not a pure causal shock",
            "next_action": "use as topology signal with warning",
        },
        {
            "question": "P4 employment signal",
            "finding": "WEAK_EXTERNAL_GAIN",
            "evidence": f"OOF residual employment locked-test MAE improvement={emp_resid.test_mae_improvement:+.6f}",
            "risk": "slope and out-of-sample gain diverge",
            "next_action": "avoid strong employment-performance claim",
        },
        {
            "question": "P4 progression signal",
            "finding": "STRONGER_THAN_EMPLOYMENT",
            "evidence": f"OOF residual D={d_resid:+.3f}%p",
            "risk": "progression reflects selection and aspiration, not only educational value",
            "next_action": "frame as progression alignment",
        },
        {
            "question": "RAW_A vs OOF residual",
            "finding": "NEAR_EQUIVALENT_IN_P4",
            "evidence": f"max delta={max_equiv_delta:.3e}, raw-resid corr={full_diag.raw_residual_corr:.3f}",
            "risk": "overclaiming residual as new independent information",
            "next_action": "keep equivalence warning in P6",
        },
        {
            "question": "P2-Q/P3-Q branch",
            "finding": "BLOCKED",
            "evidence": p6_status["P6_Q_BRANCH_STATUS"],
            "risk": "admission/selectivity feature contract not approved",
            "next_action": "do not merge Q branch into confirmatory result",
        },
    ]
)
decision_path = OUT_ROOT / "artifacts/P2_G6_1_VISUAL_DECISION_MATRIX.csv"
decision_matrix.to_csv(decision_path, index=False)

score_map = {"READY": 1.0, "STRONGER_THAN_EMPLOYMENT": 0.75, "WEAK_EXTERNAL_GAIN": 0.45, "NEAR_EQUIVALENT_IN_P4": 0.5, "BLOCKED": 0.0}
fig, ax = plt.subplots(figsize=(11, 4.8))
scores = decision_matrix["finding"].map(score_map).fillna(0.25)
colors = ["#59A14F" if s >= 0.75 else "#F58518" if s >= 0.45 else "#E15759" for s in scores]
ax.barh(decision_matrix["question"], scores, color=colors)
ax.set_xlim(0, 1)
ax.set_xlabel("decision readiness / interpretive strength")
ax.set_title("P2-G6 model-reading decision dashboard")
for y, row in enumerate(decision_matrix.itertuples(index=False)):
    ax.text(0.02, y, row.finding, va="center", ha="left", color="black", fontsize=9)
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
path = save_visual_figure(
    fig,
    "P2_G6_1_V08_P6_DECISION_DASHBOARD.png",
    "V08_P6_DECISION_DASHBOARD",
    "P6로 넘길 결과론적 판단과 경고를 한 장으로 고정할 수 있는가?",
    "P3/P4 summary objects, P6 status dictionary",
)
plt.show()
print(f"saved visual: {path.relative_to(ROOT)}")
display(decision_matrix)

display_reading_note(
    "V08 해석",
    "P6 진입은 가능하지만 employment claim, residual 독립성, Q branch는 모두 제한을 달고 간다.",
    "모형 결과는 한 숫자가 아니라 handoff-ready signal과 blocked branch가 공존하는 상태다.",
    "dashboard score는 의사결정 표시일 뿐 통계량이 아니다.",
    "다음 노트북은 residual topology를 탐색하되 confirmatory wording은 P4 strict 결과에 묶는다.",
)

saved visual: workbook/p2/P2_6/figures/P2_G6_1_V08_P6_DECISION_DASHBOARD.png


/tmp/ipykernel_183879/2788284107.py:63: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,question,finding,evidence,risk,next_action
0,P3 residual handoff,READY,"coverage=7,592, coverage_rate=1.000",locked-test R2 is low; residual is not a pure ...,use as topology signal with warning
1,P4 employment signal,WEAK_EXTERNAL_GAIN,OOF residual employment locked-test MAE improv...,slope and out-of-sample gain diverge,avoid strong employment-performance claim
2,P4 progression signal,STRONGER_THAN_EMPLOYMENT,OOF residual D=+1.022%p,"progression reflects selection and aspiration,...",frame as progression alignment
3,RAW_A vs OOF residual,NEAR_EQUIVALENT_IN_P4,"max delta=7.582e-10, raw-resid corr=0.927",overclaiming residual as new independent infor...,keep equivalence warning in P6
4,P2-Q/P3-Q branch,BLOCKED,BLOCKED_FEATURE_CONTRACT,admission/selectivity feature contract not app...,do not merge Q branch into confirmatory result


**V08 해석**

- 관찰: P6 진입은 가능하지만 employment claim, residual 독립성, Q branch는 모두 제한을 달고 간다.
- 원인: 모형 결과는 한 숫자가 아니라 handoff-ready signal과 blocked branch가 공존하는 상태다.
- 제한: dashboard score는 의사결정 표시일 뿐 통계량이 아니다.
- 결론: 다음 노트북은 residual topology를 탐색하되 confirmatory wording은 P4 strict 결과에 묶는다.


In [20]:
# 이 셀은 P6 요약 그림을 생성한다. 그림은 해석 보조용이며 모형을 새로 적합하지 않는다.
plot_df = frac_display.copy()
plot_df["label"] = plot_df["outcome"].map(
    {"HEALTH_EMPLOYMENT": "Employment", "GRAD_SCHOOL_PROGRESSION": "Grad progression"}
) + "\n" + plot_df["grade_signal"].map({"RAW_A": "RAW_A", "OOF_RESIDUAL_FULL": "OOF residual"})

fig, ax = plt.subplots(figsize=(9, 4.8))
colors = ["#4C78A8" if "Employment" in label else "#F58518" for label in plot_df["label"]]
ax.bar(plot_df["label"], plot_df["ame_pp_10pp"], color=colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("AME per 10pp grade signal (percentage points)")
ax.set_title("P4 Strict Primary Slopes Read by P2-G6_1")
ax.tick_params(axis="x", rotation=0)
for idx, value in enumerate(plot_df["ame_pp_10pp"]):
    ax.text(idx, value + (0.05 if value >= 0 else -0.05), f"{value:+.2f}", ha="center", va="bottom" if value >= 0 else "top")
fig.tight_layout()
figure_path = OUT_ROOT / "figures/P2_G6_1_P4_AME_SUMMARY.png"
fig.savefig(figure_path, dpi=160)
plt.show()
print(f"saved figure: {figure_path.relative_to(ROOT)}")


saved figure: workbook/p2/P2_6/figures/P2_G6_1_P4_AME_SUMMARY.png


/tmp/ipykernel_183879/877590253.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [21]:
# 이 셀은 P6 요약 artifact와 manifest를 저장한다.
key_numbers = pd.DataFrame(
    [
        {"metric": "P3_FULL_OOF_MAE", "value": full_oof.oof_mae, "unit": "a_rate_pct point"},
        {"metric": "P3_FULL_OOF_R2", "value": full_oof.oof_r2, "unit": "R2"},
        {"metric": "P3_FULL_TEST_R2", "value": full_oof.test_r2, "unit": "R2"},
        {"metric": "P3_FULL_RAW_RESIDUAL_CORR", "value": full_diag.raw_residual_corr, "unit": "correlation"},
        {"metric": "P3_FULL_RESIDUAL_TO_RAW_VARIANCE_RATIO", "value": full_diag.residual_to_raw_variance_ratio, "unit": "ratio"},
        {"metric": "P4_RAW_A_EMPLOYMENT_AME_PP", "value": frac_display.loc[(frac_display.outcome == "HEALTH_EMPLOYMENT") & (frac_display.grade_signal == "RAW_A"), "ame_pp_10pp"].iloc[0], "unit": "percentage point"},
        {"metric": "P4_RAW_A_PROGRESSION_AME_PP", "value": frac_display.loc[(frac_display.outcome == "GRAD_SCHOOL_PROGRESSION") & (frac_display.grade_signal == "RAW_A"), "ame_pp_10pp"].iloc[0], "unit": "percentage point"},
        {"metric": "P4_RESID_EMPLOYMENT_AME_PP", "value": frac_display.loc[(frac_display.outcome == "HEALTH_EMPLOYMENT") & (frac_display.grade_signal == "OOF_RESIDUAL_FULL"), "ame_pp_10pp"].iloc[0], "unit": "percentage point"},
        {"metric": "P4_RESID_PROGRESSION_AME_PP", "value": frac_display.loc[(frac_display.outcome == "GRAD_SCHOOL_PROGRESSION") & (frac_display.grade_signal == "OOF_RESIDUAL_FULL"), "ame_pp_10pp"].iloc[0], "unit": "percentage point"},
        {"metric": "P4_RESID_D_PP", "value": d_resid, "unit": "percentage point"},
    ]
)
key_numbers_path = OUT_ROOT / "artifacts/P2_G6_1_KEY_NUMBERS.csv"
status_path = OUT_ROOT / "qa/P2_G6_1_STATUS_TABLE.csv"
report_path = OUT_ROOT / "reports/P2_G6_1_STRICT_CHAIN_RUNUP_REPORT.md"
manifest_path = OUT_ROOT / "logs/P2_G6_1_OUTPUT_MANIFEST.json"

key_numbers.to_csv(key_numbers_path, index=False)
df_p6_status.to_csv(status_path, index=False)

status_markdown = "| status_key | status |\n|---|---|\n" + "\n".join(
    f"| {row.status_key} | {row.status} |" for row in df_p6_status.itertuples(index=False)
)

report_md = f"""# P2-G6_1 Strict P3→P4 Run-up Report

## 한 줄 판정

P3-S OOF residual과 P4 strict confirmatory 결과는 생성 완료이며, P5 residual 재실행과 P6 residual topology 검토로 넘어갈 수 있다. 단, P2-Q/P3-Q는 feature approval 때문에 차단 상태이고, RAW_A와 OOF residual은 현재 선형 P4 설계에서 거의 같은 added-information 차원을 제공한다.

## Lineage

- strict D08 SHA256: `{integrated_status.get("strict_d08_sha256")}`
- P2→P3 handoff SHA256: `{integrated_status.get("p2_handoff_sha256")}`
- P3 FULL residual SHA256: `{p3_status.get("p3_full_residual_sha256")}`
- P4→P5 handoff SHA256: `{file_record(PATHS["p4_to_p5"])["sha256"]}`

## P3-S residual

- FULL OOF MAE: `{full_oof.oof_mae:.3f}`
- FULL OOF R²: `{full_oof.oof_r2:.3f}`
- FULL locked test R²: `{full_oof.test_r2:.3f}`
- residual coverage: `{int(full_diag.coverage_n):,}` / `{int(full_diag.coverage_n):,}` rows
- raw A-residual corr: `{full_diag.raw_residual_corr:.3f}`
- residual/raw variance ratio: `{full_diag.residual_to_raw_variance_ratio:.3f}`

## P4 primary slopes

- RAW_A employment AME: `{frac_display.loc[(frac_display.outcome == "HEALTH_EMPLOYMENT") & (frac_display.grade_signal == "RAW_A"), "ame_pp_10pp"].iloc[0]:+.3f}%p`
- RAW_A progression AME: `{frac_display.loc[(frac_display.outcome == "GRAD_SCHOOL_PROGRESSION") & (frac_display.grade_signal == "RAW_A"), "ame_pp_10pp"].iloc[0]:+.3f}%p`
- OOF residual employment AME: `{frac_display.loc[(frac_display.outcome == "HEALTH_EMPLOYMENT") & (frac_display.grade_signal == "OOF_RESIDUAL_FULL"), "ame_pp_10pp"].iloc[0]:+.3f}%p`
- OOF residual progression AME: `{frac_display.loc[(frac_display.outcome == "GRAD_SCHOOL_PROGRESSION") & (frac_display.grade_signal == "OOF_RESIDUAL_FULL"), "ame_pp_10pp"].iloc[0]:+.3f}%p`

## Employment vs progression difference

- RAW_A D: `{d_raw:+.3f}%p`
- OOF residual D: `{d_resid:+.3f}%p`
- bootstrap residual D 95% CI: `[{d_boot.ci_low_pp:+.3f}, {d_boot.ci_high_pp:+.3f}]%p`

## Warnings

- 건강보험 취업 locked test improvement는 약하거나 없다.
- 대학원 진학 locked test improvement는 더 뚜렷하다.
- P4 bootstrap은 generated-regressor를 반복 생성하지만 속도를 위해 fixed encoder와 fast least-squares를 사용한 approximation이다.
- RAW_A와 OOF residual은 같은 P4 controls를 넣는 선형 설계에서 거의 등가이므로 residual만의 별도 효과처럼 과장하지 않는다.
- P2-Q/P3-Q는 feature approval 전까지 차단한다.

## P6 status

{status_markdown}
"""
report_path.write_text(report_md, encoding="utf-8")

outputs = [
    key_numbers_path,
    status_path,
    report_path,
    figure_path,
    manifest_path,
]
inputs = [
    PATHS["integrated_status"],
    PATHS["p3_status"],
    PATHS["p3_manifest"],
    PATHS["p3_full_residual"],
    PATHS["p4_status"],
    PATHS["p4_manifest"],
    PATHS["p4_coef"],
    PATHS["p4_to_p5"],
]
manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "environment": ENV,
    "notebook_path": str(NOTEBOOK_PATH.relative_to(ROOT)),
    "statuses": p6_status,
    "lineage_inputs": [file_record(p) for p in inputs],
    "outputs": [file_record(p) for p in outputs if p != manifest_path],
}
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

display(key_numbers)
display(df_p6_status)
display(Markdown(f"Saved report: `{report_path.relative_to(ROOT)}`  \nSaved manifest: `{manifest_path.relative_to(ROOT)}`"))


,metric,value,unit
0,P3_FULL_OOF_MAE,10.473046,a_rate_pct point
1,P3_FULL_OOF_R2,0.056153,R2
2,P3_FULL_TEST_R2,-0.066762,R2
3,P3_FULL_RAW_RESIDUAL_CORR,0.926507,correlation
4,P3_FULL_RESIDUAL_TO_RAW_VARIANCE_RATIO,0.943843,ratio
5,P4_RAW_A_EMPLOYMENT_AME_PP,0.616300,percentage point
6,P4_RAW_A_PROGRESSION_AME_PP,1.725986,percentage point
7,P4_RESID_EMPLOYMENT_AME_PP,0.635722,percentage point
8,P4_RESID_PROGRESSION_AME_PP,1.657726,percentage point
9,P4_RESID_D_PP,1.022005,percentage point


,status_key,status
0,P6_INPUT_STATUS,READY
1,P6_P3_RESIDUAL_HANDOFF_STATUS,READY
2,P6_P4_CONFIRMATORY_STATUS,READY_WITH_WARNINGS
3,P6_P5_RESIDUAL_RERUN_STATUS,READY
4,P6_RESIDUAL_TOPOLOGY_STATUS,READY_WITH_WARNINGS
5,P6_Q_BRANCH_STATUS,BLOCKED_FEATURE_CONTRACT
6,P6_OVERALL_STATUS,READY_WITH_WARNINGS


Saved report: `workbook/p2/P2_6/reports/P2_G6_1_STRICT_CHAIN_RUNUP_REPORT.md`  
Saved manifest: `workbook/p2/P2_6/logs/P2_G6_1_OUTPUT_MANIFEST.json`

## V09. Visual Artifact Manifest / Reading Guide

- 시각화 목적: 추가된 그림과 해석 가이드를 후속 로컬 에이전트가 그대로 사용할 수 있게 고정한다.
- 사용할 데이터: `VISUAL_FIGURE_RECORDS`, decision matrix.
- 필요한 전처리: figure record를 표로 저장하고 모델 읽기 가이드를 markdown으로 작성한다.
- 코드 셀 설계: `P2_G6_1_VISUAL_ARTIFACTS.csv`, `P2_G6_1_VISUAL_MODEL_READING_GUIDE.md` 저장.
- 그래프 해석 포인트: 시각화는 최종 claim이 아니라 claim 안전장치다.
- 학생이 자주 하는 오해: 예쁜 그림이 많을수록 분석이 강해진다고 생각하는 것.
- 체크포인트 질문: 각 그림은 어떤 모델 질문에 답하는가?

In [22]:
# V09. 시각화 산출물 manifest와 모델 읽기 가이드를 저장한다.
visual_records = pd.DataFrame(VISUAL_FIGURE_RECORDS)
visual_manifest_path = OUT_ROOT / "artifacts/P2_G6_1_VISUAL_ARTIFACTS.csv"
visual_guide_path = OUT_ROOT / "reports/P2_G6_1_VISUAL_MODEL_READING_GUIDE.md"
visual_records.to_csv(visual_manifest_path, index=False)

try:
    visual_records_table = visual_records.to_markdown(index=False)
except Exception:
    visual_records_table = "```csv\n" + visual_records.to_csv(index=False) + "```"

visual_guide_md = f"""# P2-G6_1 Visual Model Reading Guide

## 목적

이 가이드는 P2-G6_1 노트북에 추가한 시각화가 어떤 모델 질문을 답하는지 고정한다.
노트북은 P3/P4를 다시 적합하지 않고, strict-clean 산출물을 읽어 P6 진입 전 판단을 구조화한다.

## 핵심 결론

1. P3 residual handoff는 준비됐지만, locked-test R2가 낮으므로 residual을 강한 예측모형의 순수 잔차처럼 해석하지 않는다.
2. P4에서 grade signal은 건강보험 취업률보다 대학원 진학률과 더 크게 정렬된다.
3. RAW_A와 OOF residual은 현재 P4 선형 설계에서 거의 같은 added-information 축을 제공한다.
4. P2-Q/P3-Q branch는 feature contract 승인 전까지 confirmatory chain에 넣지 않는다.

## 그림 목록

{visual_records_table}

## 구조화 담론

| 항목 | 관찰 | 원인 | 제한 | 결론 |
|---|---|---|---|---|
| P3 residual | coverage는 충분하지만 raw A와 residual 상관이 높다 | residual이 raw A에서 구조 기대값을 뺀 값이기 때문 | causal shock 아님 | topology signal로만 사용 |
| P4 employment | locked-test gain이 약하다 | 취업 outcome은 구조 통제 후 grade signal 추가정보가 제한적 | slope와 예측개선이 다름 | 강한 취업성과 claim 금지 |
| P4 progression | D가 양수 방향이다 | grade signal이 대학원 진학과 더 정렬 | selection/aspiration 혼재 | progression alignment로 표현 |
| RAW_A/residual | end-to-end delta가 극소다 | 같은 controls 안에서 같은 1차원 정보를 span | 현재 P4 선형 설계 한정 | residual 고유효과 과장 금지 |
"""
visual_guide_path.write_text(visual_guide_md, encoding="utf-8")
display(visual_records)
display(Markdown(f"Saved visual manifest: `{visual_manifest_path.relative_to(ROOT)}`  \nSaved visual guide: `{visual_guide_path.relative_to(ROOT)}`"))

,block_id,figure_path,question,data_used
0,V01_STATUS_LINEAGE,workbook/p2/P2_6/figures/P2_G6_1_V01_STATUS_LI...,P3/P4 strict 상태와 hash가 같은 실행 사슬을 가리키는가?,"P3_P4_CONFIRMATORY_STATUS.json, P3/P4 status json"
1,V02_P3_RESIDUAL_DIAGNOSTIC,workbook/p2/P2_6/figures/P2_G6_1_V02_P3_RESIDU...,"P3 residual은 강한 예측모형의 잔차인가, 구조 통제 후 남은 grade s...","P3_STRUCTURE_GRADE_RESIDUAL_FULL.parquet, P3_O..."
2,V03_P4_SAMPLE_STRUCTURE,workbook/p2/P2_6/figures/P2_G6_1_V03_P4_SAMPLE...,P4 모형의 비교가 같은 표본 구조와 outcome availability 위에서 ...,"P4_SAMPLE_AUDIT.csv, P4_STRUCTURE_JOINT_FRAME...."
3,V04_P4_AME_FOREST,workbook/p2/P2_6/figures/P2_G6_1_V04_P4_AME_FO...,RAW_A와 OOF residual은 취업률/대학원 진학률에 어느 정도의 추가 신호...,"P4_COEFFICIENT_RESULTS.csv, P4_BOOTSTRAP_CI.csv"
4,V05_D_BOOTSTRAP,workbook/p2/P2_6/figures/P2_G6_1_V05_D_BOOTSTR...,grade signal의 추가 효과가 취업률보다 대학원 진학률에서 더 큰가?,"P4_EMPLOYMENT_PROGRESSION_DIFFERENCE.csv, P4_B..."
5,V06_LOCKED_TEST_IMPROVEMENT,workbook/p2/P2_6/figures/P2_G6_1_V06_LOCKED_TE...,P4 signal 추가가 locked test에서 base 대비 외부 예측을 개선했는가?,P4_LOCKED_TEST_METRICS.csv
6,V07_RAW_RESID_EQUIVALENCE,workbook/p2/P2_6/figures/P2_G6_1_V07_RAW_RESID...,현재 P4 설계에서 RAW_A와 OOF residual을 독립적인 두 신호처럼 해석...,"P4_WITHIN_RAW_EQUIVALENCE_AUDIT.csv, P3_STRUCT..."
7,V08_P6_DECISION_DASHBOARD,workbook/p2/P2_6/figures/P2_G6_1_V08_P6_DECISI...,P6로 넘길 결과론적 판단과 경고를 한 장으로 고정할 수 있는가?,"P3/P4 summary objects, P6 status dictionary"


Saved visual manifest: `workbook/p2/P2_6/artifacts/P2_G6_1_VISUAL_ARTIFACTS.csv`  
Saved visual guide: `workbook/p2/P2_6/reports/P2_G6_1_VISUAL_MODEL_READING_GUIDE.md`

In [23]:
# 최종 상태를 노트북 출력으로 분리해 둔다.
print("P6_INPUT_STATUS =", p6_status["P6_INPUT_STATUS"])
print("P6_P3_RESIDUAL_HANDOFF_STATUS =", p6_status["P6_P3_RESIDUAL_HANDOFF_STATUS"])
print("P6_P4_CONFIRMATORY_STATUS =", p6_status["P6_P4_CONFIRMATORY_STATUS"])
print("P6_P5_RESIDUAL_RERUN_STATUS =", p6_status["P6_P5_RESIDUAL_RERUN_STATUS"])
print("P6_RESIDUAL_TOPOLOGY_STATUS =", p6_status["P6_RESIDUAL_TOPOLOGY_STATUS"])
print("P6_Q_BRANCH_STATUS =", p6_status["P6_Q_BRANCH_STATUS"])
print("P6_OVERALL_STATUS =", p6_status["P6_OVERALL_STATUS"])


P6_INPUT_STATUS = READY
P6_P3_RESIDUAL_HANDOFF_STATUS = READY
P6_P4_CONFIRMATORY_STATUS = READY_WITH_WARNINGS
P6_P5_RESIDUAL_RERUN_STATUS = READY
P6_RESIDUAL_TOPOLOGY_STATUS = READY_WITH_WARNINGS
P6_Q_BRANCH_STATUS = BLOCKED_FEATURE_CONTRACT
P6_OVERALL_STATUS = READY_WITH_WARNINGS
